In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/MedAssist-AI/backend/ml/datasets/new_dataset.csv")
df

,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189642,open wound of the nose,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
189643,open wound of the nose,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
189644,open wound of the nose,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
189645,open wound of the nose,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
counts = df["diseases"].value_counts()

df = df[df["diseases"].isin(counts[counts >=200].index)]

In [ ]:
X = df.drop(columns=["diseases"])

y = df["diseases"]


print(X.shape)
print(y.shape)

(166460, 377)
(166460,)


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)


print(len(label_encoder.classes_))

265


In [ ]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)


print(X_train.shape)
print(X_test.shape)

(133168, 377)
(33292, 377)


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.8 MB/s eta 0:00:00


In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

rf_tuned = RandomForestClassifier(
    n_estimators=500,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    max_depth=35,
    bootstrap=True,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)



xgb_tuned = XGBClassifier(
    objective="multi:softmax",
    num_class=len(label_encoder.classes_),
    n_estimators=500,
    learning_rate=0.03,
    max_depth=8,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)



et = ExtraTreesClassifier(
    n_estimators=200,
    max_depth=25,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

lgbm = LGBMClassifier(
    objective="multiclass",
    num_class=len(label_encoder.classes_),

    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=64,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=0.1,

    random_state=42,
    n_jobs=-1
)
cat = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=8,

    loss_function="MultiClass",

    random_seed=42,
    verbose = 100,
    thread_count=-1
)


In [ ]:
rf_tuned.fit(X_train,y_train)

RandomForestClassifier(class_weight='balanced', max_depth=35,
                       min_samples_leaf=2, min_samples_split=5,
                       n_estimators=500, n_jobs=-1, random_state=42)

In [ ]:
xgb_tuned.fit(X_train,y_train)

In [ ]:
lgbm.fit(X_train,y_train)

In [ ]:
cat.fit(X_train,y_train)

In [ ]:
et.fit(X_train,y_train)

In [ ]:
models = {
    "Random Forest": rf_tuned,
    "XGBoost": xgb_tuned,
    "Extra Trees": et,
    "LightGBM": lgbm,
    "Catboost":cat
}

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

results = {}

for name, model in models.items():

    print("="*60)
    print(name)
    print("="*60)
    # Predict
    pred = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test,pred)

    f1 = f1_score(y_test,pred,average="weighted")

    results[name] = {
        "Accuracy": acc,
        "F1 Score": f1
    }
    print(f"{name} Accuracy : {acc:.4f}")
    print(f"{name} F1 Score  : {f1:.4f}")

In [ ]:
# Final comparison
results_df = pd.DataFrame(results).T

print("="*60)
print("FINAL MODEL COMPARISON")
print("="*60)

print(results_df.sort_values(
    "Accuracy",
    ascending=False
))